In [2]:
# LAB | SQL Subqueries - Sakila Database
# Python version for Jupyter Notebook execution

import mysql.connector
import pandas as pd
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Database connection setup (adjust these parameters for your database)
# You'll need to replace these with your actual database credentials
DB_CONFIG = {
    'host': 'localhost',
    'user': 'your_username',
    'password': 'your_password',
    'database': 'sakila'
}

# Alternative: If using SQLite version of Sakila
# import sqlite3
# conn = sqlite3.connect('sakila.db')

def execute_query(query, description):
    """Execute a SQL query and return results as a pandas DataFrame"""
    try:
        # Using MySQL connector
        conn = mysql.connector.connect(**DB_CONFIG)
        df = pd.read_sql(query, conn)
        conn.close()
        
        print(f"\n{'='*60}")
        print(f"QUERY: {description}")
        print(f"{'='*60}")
        print(f"SQL: {query}")
        print(f"\nRESULTS:")
        print(df)
        print(f"Rows returned: {len(df)}")
        return df
        
    except Exception as e:
        print(f"Error executing query: {e}")
        return None

# =====================================================
# MAIN CHALLENGES
# =====================================================

# 1. Determine the number of copies of the film "Hunchback Impossible" that exist in the inventory system.
query1 = """
SELECT COUNT(*) as number_of_copies
FROM inventory i
JOIN film f ON i.film_id = f.film_id
WHERE f.title = 'Hunchback Impossible'
"""

execute_query(query1, "1. Number of copies of 'Hunchback Impossible'")

# Alternative using subquery:
query1_alt = """
SELECT COUNT(*) as number_of_copies
FROM inventory
WHERE film_id = (
    SELECT film_id 
    FROM film 
    WHERE title = 'Hunchback Impossible'
)
"""

execute_query(query1_alt, "1. Alternative - Using subquery")

# 2. List all films whose length is longer than the average length of all the films in the Sakila database.
query2 = """
SELECT title, length
FROM film
WHERE length > (
    SELECT AVG(length) 
    FROM film
)
ORDER BY length DESC
"""

execute_query(query2, "2. Films longer than average length")

# 3. Use a subquery to display all actors who appear in the film "Alone Trip".
query3 = """
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.actor_id IN (
    SELECT fa.actor_id
    FROM film_actor fa
    JOIN film f ON fa.film_id = f.film_id
    WHERE f.title = 'Alone Trip'
)
"""

execute_query(query3, "3. Actors in 'Alone Trip'")

# =====================================================
# BONUS CHALLENGES
# =====================================================

# 1. Identify all movies categorized as family films.
bonus1_joins = """
SELECT f.title
FROM film f
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
WHERE c.name = 'Family'
ORDER BY f.title
"""

execute_query(bonus1_joins, "Bonus 1. Family films (using joins)")

bonus1_subquery = """
SELECT title
FROM film
WHERE film_id IN (
    SELECT fc.film_id
    FROM film_category fc
    JOIN category c ON fc.category_id = c.category_id
    WHERE c.name = 'Family'
)
ORDER BY title
"""

execute_query(bonus1_subquery, "Bonus 1. Family films (using subquery)")

# 2. Retrieve the name and email of customers from Canada using both subqueries and joins.

# Using JOINS:
bonus2_joins = """
SELECT c.first_name, c.last_name, c.email
FROM customer c
JOIN address a ON c.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
WHERE co.country = 'Canada'
"""

execute_query(bonus2_joins, "Bonus 2. Canadian customers (using joins)")

# Using SUBQUERIES:
bonus2_subqueries = """
SELECT first_name, last_name, email
FROM customer
WHERE address_id IN (
    SELECT address_id
    FROM address
    WHERE city_id IN (
        SELECT city_id
        FROM city
        WHERE country_id = (
            SELECT country_id
            FROM country
            WHERE country = 'Canada'
        )
    )
)
"""

execute_query(bonus2_subqueries, "Bonus 2. Canadian customers (using subqueries)")

# 3. Find the most prolific actor first
most_prolific_actor = """
SELECT actor_id, COUNT(*) as film_count
FROM film_actor
GROUP BY actor_id
ORDER BY film_count DESC
LIMIT 1
"""

execute_query(most_prolific_actor, "Bonus 3. Most prolific actor")

# Films starred by the most prolific actor
bonus3 = """
SELECT f.title
FROM film f
JOIN film_actor fa ON f.film_id = fa.film_id
WHERE fa.actor_id = (
    SELECT actor_id
    FROM film_actor
    GROUP BY actor_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
ORDER BY f.title
"""

execute_query(bonus3, "Bonus 3. Films by most prolific actor")

# With actor name included:
bonus3_with_name = """
SELECT a.first_name, a.last_name, f.title
FROM film f
JOIN film_actor fa ON f.film_id = fa.film_id
JOIN actor a ON fa.actor_id = a.actor_id
WHERE fa.actor_id = (
    SELECT actor_id
    FROM film_actor
    GROUP BY actor_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
ORDER BY f.title
"""

execute_query(bonus3_with_name, "Bonus 3. Films by most prolific actor (with actor name)")

# 4. Find the most profitable customer
most_profitable_customer = """
SELECT customer_id, SUM(amount) as total_spent
FROM payment
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 1
"""

execute_query(most_profitable_customer, "Bonus 4. Most profitable customer")

# Films rented by most profitable customer:
bonus4 = """
SELECT DISTINCT f.title
FROM film f
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
WHERE r.customer_id = (
    SELECT customer_id
    FROM payment
    GROUP BY customer_id
    ORDER BY SUM(amount) DESC
    LIMIT 1
)
ORDER BY f.title
"""

execute_query(bonus4, "Bonus 4. Films rented by most profitable customer")

# 5. Average spending per customer
avg_spending = """
SELECT AVG(customer_total) as avg_spent_per_customer
FROM (
    SELECT customer_id, SUM(amount) as customer_total
    FROM payment
    GROUP BY customer_id
) as customer_totals
"""

execute_query(avg_spending, "Bonus 5. Average spending per customer")

# Customers who spent more than average:
bonus5 = """
SELECT customer_id, total_amount_spent
FROM (
    SELECT customer_id, SUM(amount) as total_amount_spent
    FROM payment
    GROUP BY customer_id
) as customer_spending
WHERE total_amount_spent > (
    SELECT AVG(customer_total)
    FROM (
        SELECT customer_id, SUM(amount) as customer_total
        FROM payment
        GROUP BY customer_id
    ) as avg_calculation
)
ORDER BY total_amount_spent DESC
"""

execute_query(bonus5, "Bonus 5. Customers spending above average")

# =====================================================
# VERIFICATION QUERIES
# =====================================================

verification1 = """
SELECT 
    COUNT(*) as total_films,
    AVG(length) as average_length,
    MIN(length) as min_length,
    MAX(length) as max_length
FROM film
"""

execute_query(verification1, "Verification: Film statistics")

verification2 = """
SELECT COUNT(*) as films_longer_than_average
FROM film
WHERE length > (SELECT AVG(length) FROM film)
"""

execute_query(verification2, "Verification: Count of films longer than average")

verification3 = """
SELECT a.first_name, a.last_name, COUNT(*) as film_count
FROM actor a
JOIN film_actor fa ON a.actor_id = fa.actor_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY film_count DESC
LIMIT 5
"""

execute_query(verification3, "Verification: Top 5 most prolific actors")

print("\n" + "="*80)
print("ALL QUERIES COMPLETED!")
print("Make sure to update the DB_CONFIG dictionary with your database credentials.")
print("="*80)

Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)
Error executing query: 2003 (HY000): Can't connect